# 47. The direct probe, on the revised streams

## Why this exists

The paper's main result is that three standard diagnostics say the model is
losing its ability to learn, while direct measurements say it is not.

The diagnostics have been re-measured on the revised design. The direct
measurements have not. They were taken on the old design of eight tasks of ten
classes, whose models the new class split cannot even load. So right now the
two halves of the central claim sit on two different experiments.

This notebook moves the most important direct measurement onto the current
streams.

## What it measures

Take a model, fork it, teach it a task it has never seen, then throw the fork
away. How well it learns that task is a direct answer to "can this model still
learn?". No proxy, no inference.

Do that at the start of the stream, before any task is learned, and again at
the end after nine. If the model has lost its ability to learn, the second
score should be lower.

The task used is group 9, the twenty classes the stream never touches. That is
cleaner than the old design, where the probe task was simply one that happened
not to be used.

## The grid, and why there is one

A model late in a stream may want a different learning rate from a fresh one.
A single fixed rate would confuse a rate mismatch with lost plasticity. So each
probe tries three rates and keeps the best, chosen on a validation split that
is never trained on.

Eleven repetitions at each point. The probe is noisy, and the floor below which
a change means nothing is built from that spread.

## What it does not change

The training budget is the same as the old passes: 4,800 images and 250 steps.
The probe task now has twenty classes instead of ten, so it takes 240 images
per class rather than 480. The model sees the same number of images either way,
which is what keeps the two points comparable to each other.

The scores themselves will be lower than the old design's 58 to 60 percent,
because twenty classes is a harder task than ten. That is expected and it does
not matter. The comparison is start against end within this design, never
against the old numbers.

## Cost

Five seeds, two points, three rates, eleven repetitions is 330 probe runs of
250 steps each. Expect a few hours on a GPU.

Every finished run is saved to R2 as it completes, so an interrupted session
resumes instead of starting again. Adding orders B and C triples the time, and
the settings cell says how.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-probe")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this from a checkout that has the ten-group split.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None and IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

import torch
from tqdm.auto import tqdm
from uncle import checkpoint
from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.hypernet import HyperNetwork, build_target
from uncle.plasticity_indicators import indicator_key
from uncle.probe import noise_floor, probe_once, snapshot, unchanged
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.trainer import UnCLe

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| device:", GPU_NAME)
if DEVICE == "cpu":
    print("No GPU. This notebook trains 330 small forks; it needs one.")

## 2. Settings

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

# Which runs to probe. order_A at five seeds is the set that carries the
# headline comparison, because its indicator records are the ones section 4.1
# reports. Adding "order_B" and "order_C" triples the session; the resume file
# means you can add them later without redoing these.
ORDERS = ["order_A"]
SEEDS = [0, 1, 2, 3, 4]

# Where in the stream to probe. 0 is the untrained generator, which the seed
# rebuilds exactly, so it needs no file. 9 is the end of the nine-task stream.
#
# Boundary 4 was also kept by every run, so adding it here measures the middle
# and costs half as much again. Boundary 10 is the model after the unlearn and
# is a different question; leave it out of this comparison.
POINTS = [0, 9]

# Names this protocol. Any change below that would make the numbers
# incomparable needs a new label, because finished runs are matched by identity
# and a shared file would quietly mix two protocols.
RUN_LABEL = "revised_20class_train4800"

RESUME_KEY = f"uncle/matched_probe/{STUDY_ID}/{RUN_LABEL}.jsonl"
SAVE_EVERY = 10

# The probe task is the held-out group. The stream never trains on it, by
# design rather than by accident, which is new in this version of the study.
PROBE_TASKS = [PROBE_TASK]
CLASSES_HINT = 20

# A model late in a stream may need a different rate than a fresh one, so the
# probe tries a grid and keeps the best on a validation split. A single fixed
# rate would confuse a rate mismatch with plasticity loss.
RATES = [1e-4, 3e-4, 1e-3]
STEPS = 250
BATCH_SIZE = 64
FEATURE_IMAGES = 2000

# Equal counts at both points, so the floor falls as one over the square root
# of the count. Eleven and eleven is what the October passes settled on.
REPETITIONS_AT_ZERO = 11
REPETITIONS_LATER = 11

# (name, generator frozen, code trains). The code-only arm was dropped in the
# old design because it read chance at both points and bought nothing.
ARMS = [("full", False, True)]

# Per class. Twenty classes at 240 images each is 4,800 training images, the
# same total the old ten-class passes used at 480 per class. The model sees the
# same number of images, so the two probe points stay comparable to each other.
# The scores will be lower than the old design's because twenty classes is a
# harder problem than ten, which does not affect the comparison.
TRAIN_PER_CLASS = 240
VAL_PER_CLASS = 20
SPLIT_SEED = 606

DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
CACHE = Path("/content/probe") if IN_COLAB else REPO / "outputs/probe-revised"
CACHE.mkdir(parents=True, exist_ok=True)

RUNS = [(order, seed) for order in ORDERS for seed in SEEDS]
print(f"{len(RUNS)} streams x {len(POINTS)} points x {len(RATES)} rates x "
      f"{REPETITIONS_AT_ZERO} repetitions = "
      f"{len(RUNS) * len(POINTS) * len(RATES) * REPETITIONS_AT_ZERO} probe runs")
print("Probe task:", PROBE_TASKS, "| points:", POINTS)

## 3. Connect to R2

In [ ]:
def secret(name):
    """Read a credential from Colab secrets, or the environment elsewhere."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    value = os.environ.get(name)
    if value is None:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    return line.partition("=")[2].strip().strip('"').strip("'")
    return value


store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))

# The settings the streams were trained with, read back rather than retyped, so
# the model this notebook rebuilds has the shape its checkpoint expects.
plan = json.loads(store.download(
    f"uncle/indicators/{STUDY_ID}/job2_summary.json", CACHE).read_text(encoding="utf-8"))
CLASSES = plan["classes_per_task"]
assert CLASSES == CLASSES_HINT, f"settings assume {CLASSES_HINT}, the runs used {CLASSES}"
print("Streams used learning rate", plan["learning_rate"], "| epochs", plan["epochs"],
      "| batch", plan["batch_size"], "| classes per task", CLASSES)

# Stop now if a stream this notebook is about to probe has no end-of-stream
# model, rather than an hour into the session.
FORGET = {"order_A": "0", "order_B": "4", "order_C": "8"}
for order, seed in RUNS:
    for point in POINTS:
        if point == 0:
            continue
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        if store.info(key) is None:
            raise RuntimeError(f"Missing {key}")
print("Every model this session needs is in R2.")

## 4. Build the images and the probe splits

In [ ]:
from torch.utils.data import Subset

STREAM_TASKS = [str(i) for i in range(9)]
needed = tuple(STREAM_TASKS + PROBE_TASKS)
tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[20],
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")

if set(PROBE_TASKS) & set(STREAM_TASKS):
    raise ValueError("The probe task is in the learning stream, so it is not held out.")


def balanced_split(labels, counts, seed):
    """Row numbers for disjoint splits, equal numbers from every class.

    Works on the labels alone, so no image is read here.
    """
    by_class = {}
    for row, label in enumerate(labels):
        by_class.setdefault(int(label), []).append(row)
    if len(by_class) != CLASSES:
        raise RuntimeError(f"Expected {CLASSES} classes, found {len(by_class)}.")
    stream = torch.Generator().manual_seed(seed)
    splits = [[] for _ in counts]
    for label, rows in sorted(by_class.items()):
        if len(rows) < sum(counts):
            raise RuntimeError(f"Class {label} has {len(rows)} images, needs {sum(counts)}.")
        order = [rows[j] for j in torch.randperm(len(rows), generator=stream).tolist()]
        at = 0
        for which, how_many in enumerate(counts):
            splits[which] += order[at:at + how_many]
            at += how_many
    taken = [row for split in splits for row in split]
    assert len(set(taken)) == len(taken), "An image landed in two splits."
    return splits


# Training and validation come out of the task's own training images, so no
# image is ever both trained on and used to pick a learning rate. The test
# split is the project's usual one.
probe_sets = {}
for task in PROBE_TASKS:
    base = tasks[task]["train"]
    train_rows, val_rows = balanced_split(base.targets,
                                          (TRAIN_PER_CLASS, VAL_PER_CLASS), SPLIT_SEED)
    probe_sets[task] = {"train": Subset(base, train_rows),
                        "val": Subset(base, val_rows),
                        "test": tasks[task]["test"]}

for name, split in probe_sets.items():
    print(f"task {name}: train {len(split['train'])}  val {len(split['val'])}  "
          f"test {len(split['test'])}")

## 5. Building a model at one probe point

In [ ]:
def build(order, seed, point):
    """The model this stream had at `point`. Point 0 needs no file."""
    settings = study_defaults(order)
    settings["requests"] = settings["requests"] + (("forget", FORGET[order]),)
    config = Config(**settings, seed=seed,
                    learning_rate=plan["learning_rate"], epochs=plan["epochs"],
                    batch_size=plan["batch_size"],
                    initialization=plan["initialization"], device=DEVICE)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if point > 0:
        key = f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}/boundary_{point}.pt"
        payload = torch.load(store.download(key, CACHE), map_location=DEVICE,
                             weights_only=False)
        checkpoint.restore(payload, hypernet=hypernet, uncle=trainer)
    return target, hypernet, trainer

## 6. Run the grid

In [ ]:
def identity(run):
    """Everything that makes one probe run different from another."""
    return (run["order"], run["seed_label"], run["point"], run["task"],
            run["learning_rate"], run["arm"], run["repetition"])


def probe_seed(task, repetition):
    """Fixes the batch order and the fresh code.

    Depends on the task and the repetition, never on the probe point, so two
    points differ only by the state of the model being probed.
    """
    return 12345 + 1000 * repetition + (int(task) if task.isdigit() else 99)


PARTIAL = CACHE / "probe_partial.jsonl"
results = []
if store.info(RESUME_KEY) is not None:
    saved = store.download(RESUME_KEY, CACHE)
    for line in saved.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipped one unreadable line at the end of the saved file.")
    print(f"Resuming: {len(results)} runs already finished.")
else:
    print("Starting fresh: nothing saved in R2 yet.")

PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results), encoding="utf-8")
done = {identity(r) for r in results}
new_runs = 0


def grid_at(order, seed, point, repetitions):
    """Every probe task, rate, arm and repetition at one point of one stream."""
    global new_runs
    todo = [(task, rate, arm, code_only, train_code, rep)
            for task in PROBE_TASKS
            for rate in RATES
            for arm, code_only, train_code in ARMS
            for rep in range(repetitions)
            if (order, seed, point, task, rate, arm, rep) not in done]
    if not todo:
        print(f"{order} seed {seed}, t = {point}: already finished")
        return

    target, hypernet, trainer = build(order, seed, point)
    reference = snapshot(hypernet, trainer)
    bar = tqdm(todo, desc=f"{order} seed {seed}  t={point}", unit="run")
    for task, rate, arm, code_only, train_code, rep in bar:
        bar.set_postfix_str(f"lr={rate:g} rep {rep + 1}")
        started = time.perf_counter()
        run = probe_once(
            hypernet, target, trainer, task=task,
            train_set=probe_sets[task]["train"], test_set=probe_sets[task]["test"],
            validation_set=probe_sets[task]["val"],
            steps=STEPS, learning_rate=rate, batch_size=BATCH_SIZE,
            code_only=code_only, train_code=train_code,
            seed=probe_seed(task, rep), feature_images=FEATURE_IMAGES)
        # A probe forks and discards. If it changed the model, every later
        # reading in this session is measuring something that has moved.
        if not unchanged(hypernet, reference):
            raise RuntimeError("A probe changed the model. Stop and fix this "
                               "before trusting anything.")
        run.update(order=order, point=point, endpoint="E2", repetition=rep,
                   seed_label=seed, minutes=(time.perf_counter() - started) / 60,
                   device=GPU_NAME, run_label=RUN_LABEL, code_revision=HEAD,
                   train_images=len(probe_sets[task]["train"]))
        results.append(run)
        done.add(identity(run))
        with PARTIAL.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(run) + "\n")
        new_runs += 1
        if new_runs % SAVE_EVERY == 0:
            store.upload(PARTIAL, RESUME_KEY, overwrite=True)
    bar.close()

    del trainer, hypernet, target
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    store.upload(PARTIAL, RESUME_KEY, overwrite=True)


began = time.perf_counter()
for order, seed in RUNS:
    for point in POINTS:
        grid_at(order, seed, point,
                REPETITIONS_AT_ZERO if point == 0 else REPETITIONS_LATER)
print(f"\nFinished in {(time.perf_counter() - began) / 3600:.1f} hours. "
      f"{len(results)} runs in total.")

## 7. The result, against what the indicators said

In [ ]:
import statistics


def select(runs):
    """The run with the best validation accuracy. Ties go to the lower rate."""
    return min(runs, key=lambda r: (-r["val_accuracy"], r["learning_rate"]))


def per_repetition(order, seed, point, task):
    """One number per repetition: the test accuracy of the selected run."""
    values = []
    for rep in range(max(REPETITIONS_AT_ZERO, REPETITIONS_LATER)):
        runs = [r for r in results
                if r["order"] == order and r["seed_label"] == seed
                and r["point"] == point and r["task"] == task
                and r["arm"] == "full" and r["repetition"] == rep]
        if runs:
            values.append(select(runs)["test_accuracy"])
    return values


# What the indicators said about these same models, so the two sit in one
# table instead of being compared by hand across two notebooks.
def rank_change(order, seed, a, b):
    key = indicator_key(STUDY_ID, order, seed)
    if store.info(key) is None:
        return None
    rows = {int(json.loads(l)["t"]): json.loads(l) for l
            in store.download(key, CACHE).read_text(encoding="utf-8").splitlines()
            if l.strip()}
    if a not in rows or b not in rows:
        return None
    start = rows[a]["effective_rank_in_stream"]
    return (rows[b]["effective_rank_in_stream"] - start) / start * 100


task = PROBE_TASKS[0]
lo, hi = POINTS[0], POINTS[-1]
print(f"Probe task {task}, {CLASSES} classes, so guessing scores "
      f"{100 / CLASSES:.0f}%")
print()
print(f"{'run':>16}{'probe t=' + str(lo):>12}{'probe t=' + str(hi):>12}"
      f"{'change':>9}{'floor':>8}{'effective rank':>16}")
rows = []
for order, seed in RUNS:
    at = {p: per_repetition(order, seed, p, task) for p in (lo, hi)}
    if not (at[lo] and at[hi]):
        print(f"{order + '_seed' + str(seed):>16}{'not finished':>12}")
        continue
    spread = statistics.stdev(at[lo]) if len(at[lo]) > 1 else float("nan")
    floor = noise_floor(spread, spread, len(at[lo]), len(at[hi]))
    change = statistics.mean(at[hi]) - statistics.mean(at[lo])
    rank = rank_change(order, seed, lo, hi)
    rows.append({"order": order, "seed": seed, "t0": statistics.mean(at[lo]),
                 "t_end": statistics.mean(at[hi]), "change": change,
                 "floor": floor, "rank_change_pct": rank})
    print(f"{order + '_seed' + str(seed):>16}{statistics.mean(at[lo]):>11.2f}%"
          f"{statistics.mean(at[hi]):>11.2f}%{change:>+9.2f}{floor:>8.2f}"
          + (f"{rank:>15.1f}%" if rank is not None else f"{'-':>16}"))

if rows:
    rose = sum(r["change"] > r["floor"] for r in rows)
    fell = sum(r["change"] < -r["floor"] for r in rows)
    print()
    print(f"Probe accuracy rose past its floor in {rose} of {len(rows)} runs, "
          f"fell past it in {fell}.")
    print(f"Mean change: {statistics.mean(r['change'] for r in rows):+.2f} points.")
    ranks = [r["rank_change_pct"] for r in rows if r["rank_change_pct"] is not None]
    if ranks:
        print(f"Effective rank over the same span: "
              f"{statistics.mean(ranks):+.1f}% on average, "
              f"falling in {sum(x < 0 for x in ranks)} of {len(ranks)} runs.")
    print()
    if fell == 0 and ranks and all(x < 0 for x in ranks):
        print("The diagnostics say the representation collapsed. The direct")
        print("measurement says the model learns a new task at least as well as")
        print("it did before any training. That is the paper's central result,")
        print("now on the revised design.")
    else:
        print("Read the two columns together before concluding anything.")

summary = {"study": STUDY_ID, "run_label": RUN_LABEL, "code_revision": HEAD,
           "probe_task": task, "classes": CLASSES, "points": [lo, hi],
           "train_images": TRAIN_PER_CLASS * CLASSES, "steps": STEPS,
           "rates": RATES, "repetitions": REPETITIONS_AT_ZERO, "rows": rows}
path = CACHE / "probe_summary.json"
path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
store.upload(path, f"uncle/matched_probe/{STUDY_ID}/{RUN_LABEL}_summary.json",
             overwrite=True)
print("Written:", path, "and uploaded to R2.")

## 8. What comes next

If probe accuracy holds or rises while effective rank falls, the central claim
is re-established on the current design and no longer depends on the retired
eight-task runs.

Three direct measurements are still on the old design after this one:

- fitting random labels, which is notebook 41's arm
- the speed of learning an unseen task, which reads the probe shapes this
  notebook already saves, so it needs no new GPU time
- curvature at a trained code, which is notebook 42's third run

The second of those is nearly free once this notebook has run, because
`probe_once` records four readings along each probe, not just the end.